# Cross-Encoder Mech-Interp Prototype\nQuestion-driven relevance analysis for e-commerce query-item pairs.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/salvatoretornatore/Dev-Sandbox/BERT-Mech-Interp')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from curate_dataset import build_probe_set
from inference import load_cross_encoder, score_pairs
from attribution import token_gradient_attribution
from attention import attention_summary
from reporting import evaluate_directional_checks, make_failure_triage, export_artifacts

In [2]:
handcrafted_csv = ROOT / 'data' / 'handcrafted_seed.csv'
probe_csv = ROOT / 'data' / 'probe_set_v1.csv'
probes_df = build_probe_set(handcrafted_csv, probe_csv, target_size=80)
probes_df.head()

,probe_id,source,question_tag,query,item_text,esci_label,relevance_score,pair_group_id,expected_direction,target_tokens_query,target_tokens_item,notes
0,h_0001,handcrafted,brand_match,nike running shoes,Nike Pegasus 40 men's running shoes,E,3,grp_h_0001,should_rank_higher,nike|running,Nike|Pegasus,exact brand match
1,h_0002,handcrafted,brand_match,nike running shoes,Adidas Duramo men's running shoes,I,0,grp_h_0001,should_rank_lower,nike|running,Adidas,brand mismatch
2,h_0003,handcrafted,brand_match,samsung 65 inch tv,Samsung 65 inch QLED 4K Smart TV,E,3,grp_h_0002,should_rank_higher,samsung|65 inch,Samsung|QLED,brand and category match
3,h_0004,handcrafted,brand_match,samsung 65 inch tv,LG 65 inch OLED Smart TV,S,2,grp_h_0002,should_rank_lower,samsung|65 inch,LG,competitor brand
4,h_0005,handcrafted,attribute_match,iphone 14 case magsafe,Case for iPhone 14 with MagSafe ring,E,3,grp_h_0003,should_rank_higher,iphone 14|magsafe,iPhone 14|MagSafe,compatibility match


In [3]:
bundle = load_cross_encoder()
scored_df = score_pairs(bundle, probes_df, batch_size=16)
scored_df[['probe_id', 'question_tag', 'score']].head()

,probe_id,question_tag,score
0,h_0001,brand_match,0.999712
1,h_0002,brand_match,0.940245
2,h_0003,brand_match,0.999864
3,h_0004,brand_match,0.998165
4,h_0005,attribute_match,0.999741


In [4]:
checks_df = evaluate_directional_checks(scored_df)
checks_df[['question_tag', 'pass_rate', 'num_groups']].drop_duplicates().sort_values('question_tag')

,question_tag,pass_rate,num_groups
10,attribute_match,1.0,3
0,brand_match,1.0,4
16,bundle_vs_canonical,1.0,3
1,negation,0.6,10


In [5]:
example = scored_df.iloc[0]
token_attr_df = token_gradient_attribution(bundle, example['query'], example['item_text'])
token_attr_df.sort_values('abs_attr', ascending=False).head(15)

,position,token,segment,signed_attr,abs_attr,norm_abs_attr
12,12,shoes,item,-0.165542,0.165542,0.193792
2,2,running,query,0.142828,0.142828,0.167203
5,5,nike,item,-0.101414,0.101414,0.118721
3,3,shoes,query,0.087272,0.087272,0.102165
1,1,nike,query,0.065455,0.065455,0.076625
4,4,[SEP],special,-0.062583,0.062583,0.073263
6,6,pegasus,item,-0.048525,0.048525,0.056806
7,7,40,item,-0.046138,0.046138,0.054011
8,8,men,item,0.041741,0.041741,0.048864
9,9,',item,0.034405,0.034405,0.040276


In [6]:
attention_df = attention_summary(bundle, example['query'], example['item_text'])
attention_df.head()

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


,layer,head,cls_to_query_mean,cls_to_item_mean,query_to_item_mean
0,0,0,0.074511,6.971604e-02,4.474935e-02
1,0,1,0.068613,7.299285e-02,5.624145e-02
2,0,2,0.200000,4.656231e-11,4.630512e-09
3,0,3,0.140978,3.278999e-02,3.954245e-02
4,0,4,0.189047,6.084798e-03,1.032895e-02


In [7]:
failure_df = make_failure_triage(scored_df, checks_df)
export_artifacts(ROOT / 'outputs', scored_df, checks_df, failure_df, token_attr_df, attention_df)
failure_df.head()

,pair_group_id,question_tag,query,item_text,esci_label,relevance_score,expected_direction,model_pred_direction,score,group_score_margin,suspected_failure_type
3,grp_0003,negation,# 2 pencils not sharpened,Emraw Pre Sharpened Triangular Primary Size No...,Irrelevant,0,should_rank_lower,predicted_higher,0.973898,0.117801,negation-handling
2,grp_0003,negation,# 2 pencils not sharpened,"iScholar Gross Pack Pencils, #2, Yellow, Box o...",Exact,3,should_rank_higher,predicted_lower,0.856097,0.117801,negation-handling
5,grp_0005,negation,#1 selling shoes for men without shoeleases,"Skechers Men's Moreno Canvas Oxford Shoe, Taup...",Irrelevant,0,should_rank_lower,predicted_higher,0.004622,0.004571,negation-handling
4,grp_0005,negation,#1 selling shoes for men without shoeleases,Skechers Men's GO Walk Evolution Ultra-Impecca...,Exact,3,should_rank_higher,predicted_lower,0.000052,0.004571,negation-handling
7,grp_0006,negation,#1 small corded treadmill without remote control,Egofit Walker Pro Smallest Under Desk Electric...,Irrelevant,0,should_rank_lower,predicted_higher,0.381395,0.380792,negation-handling
